# Vector Databases, FAISS & Chroma - Complete Playbook

This notebook is a hands-on guide to:
1. **Text Embeddings** - converting words/sentences into numbers
2. **Similarity Search** - finding "semantically similar" text
3. **FAISS** - Facebook AI Similarity Search library
4. **LangChain Vector Stores** - easy wrappers around FAISS/Chroma
5. **ChromaDB** - another popular vector database

Run the cells top to bottom. Each section explains what is happening and why.


## Section 1: Environment Setup

We use a `.env` file to store API keys safely. The notebook reads `HF_TOKEN` (HuggingFace) and `OPENAI_API_KEY` if you want to use OpenAI models later.

The `load_dotenv()` function loads these keys into environment variables. The `os.getenv("KEY", "")` pattern prevents crashes if the key is missing.


In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

In [ ]:
import os
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")


## Section 2: Text Embeddings

An **embedding** is a numerical representation of text. Similar meanings get similar vectors.

We use `all-MiniLM-L6-v2`, a free, local model that produces 384-dimensional vectors. Every sentence, whether short or long, becomes a vector of exactly 384 numbers.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")

**Embed a single sentence.**

`embed_query()` converts one string into a vector. We check `len(embed)` to confirm it has 384 dimensions.


In [ ]:
embed = embeddings.embed_query("Hello AI")

In [ ]:
len(embed)

**Try a longer sentence.**

Notice that even with more words, the embedding length is still 384. The model compresses meaning into a fixed-size vector.


In [ ]:
embed = embeddings.embed_query("Hello AI, this is genai concepts")

In [ ]:
len(embed)

## Section 3: Comparing Sentences

Once text is converted to vectors, we can compare them mathematically.

Two popular metrics:
- **Cosine Similarity**: measures the angle between vectors. Range [-1, 1]. Higher = more similar.
- **Euclidean (L2) Distance**: measures straight-line distance. Range [0, ∞). Lower = more similar.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

**Create sample documents and a query.**

We will compare the query against each document to see which one is most similar.


In [ ]:
documents = [
             "What is the capital of USA?",
             "Who is the president of USA?",
             "WHo is prime minister of India?"
            ]

In [ ]:
my_query = "Narendra Modi is the prime minister of India"

**Embed the documents and the query.**

`embed_documents()` handles a list of strings. We then check shapes to confirm:
- 3 documents
- each document embedding has 384 dimensions
- query embedding also has 384 dimensions


In [ ]:
document_embeddings = embeddings.embed_documents(documents)

In [ ]:
len(document_embeddings)

In [ ]:
len(document_embeddings[0]) # ---> first stateent

In [ ]:
len(document_embeddings[1])

In [ ]:
len(document_embeddings[2])

In [ ]:
query_embeddings = embeddings.embed_query(my_query)

In [ ]:
len(query_embeddings)

**Compute cosine similarity.**

The query is about India's prime minister. Expect the 3rd document ("Who is prime minister of India?") to have the highest score.


In [ ]:
cosine_similarity([query_embeddings],document_embeddings)

**Try a different query.**

"Leader of the countries" is more generic. See how the scores change across documents.


In [ ]:
my_query = "Leader of the countries"
query_embeddings = embeddings.embed_query(my_query)
cosine_similarity([query_embeddings],document_embeddings)

**Compute Euclidean (L2) distance.**

For L2 distance, smaller numbers mean more similar. Compare these values with the cosine similarity values above.


In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

In [ ]:
euclidean_distances([query_embeddings], document_embeddings)

In [ ]:
| Metric            | Similarity Score Range | Behavior                              |
| ----------------- | ---------------------- | ------------------------------------- |
| Cosine Similarity | \[-1, 1]               | Focuses on angle only |
| L2 Distance       | \[0, ∞)                | Focuses on **magnitude + direction**  |


## Section 4: FAISS - Facebook AI Similarity Search

FAISS is a library for fast nearest-neighbor search on vector data. It is especially useful when you have thousands or millions of embeddings.

We will:
1. Create a FAISS index
2. Add text embeddings to it
3. Search for similar text
4. Inspect how LangChain maps FAISS IDs back to text


In [ ]:
import faiss
import chromadb
from langchain_community.vectorstores import FAISS, Chroma
from langchain_community.docstore.in_memory import InMemoryDocstore





**Create a FAISS index.**

`IndexFlatL2` is the simplest FAISS index. It does an exact search using L2 (Euclidean) distance. The number 384 matches our embedding dimension.


In [ ]:
# Index first


index = faiss.IndexFlatL2(384)

In [ ]:
index

**Wrap the index in a LangChain FAISS vector store.**

LangChain's `FAISS` class needs:
- `embedding_function`: how to turn text into vectors
- `index`: the FAISS index
- `docstore`: stores the actual text, mapped by ID
- `index_to_docstore_id`: empty at first, filled as we add data


In [ ]:
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id = {},
)

In [ ]:
vector_store

**Add text strings to the vector store.**

`add_texts()` embeds each string and stores both the vector (in FAISS) and the text (in the docstore).


In [ ]:
vector_store.add_texts(["AI is future", "AI is powerful", "Dogs are cute", "Pizza is good", "I like mexican food"])

**Inspect the ID mapping.**

FAISS uses integer positions (0, 1, 2...). The docstore uses UUIDs. This dictionary maps positions to UUIDs.


In [ ]:
vector_store.index_to_docstore_id

**Look up one specific entry.**

Pick FAISS index 1, find its docstore UUID, then retrieve the original text. This shows how vectors and text stay linked.


In [ ]:
faiss_index = 1
docstore_id = vector_store.index_to_docstore_id[faiss_index]
print(docstore_id)


In [ ]:
vector_store.docstore.search(docstore_id)

**Search the vector store.**

`similarity_search("Tell me about AI", k=2)` converts the query to a vector, finds the 2 closest vectors in FAISS, and returns their text.


In [ ]:
results = vector_store.similarity_search("Tell me about AI",k=2)

# Tell me about AI ---> embeddings (embedding model)
# Configurations in Index --> similarity
# Ordr them and filter them and give us k documents

In [ ]:
results

**Experiment with different queries and k values.**

`k` controls how many results come back. Try changing it to see more or fewer matches.


In [ ]:
results = vector_store.similarity_search("I love cat",k=2)

In [ ]:
results

In [ ]:
results = vector_store.similarity_search("I love cat",k=5)

In [ ]:
results

In [ ]:
results = vector_store.similarity_search("My favourite super hero",k=5)
results

In [ ]:
| Feature               | `Flat`                | `IVF` (Inverted File Index)        | `HNSW` (Graph-based Index)          |
| --------------------- | --------------------- | ---------------------------------- | ----------------------------------- |
| Type of Search     | Exact                 | Approximate (cluster-based)        | Approximate (graph-based traversal) |
| Speed               | Slow (linear scan)    | Fast (search only in top clusters) | Very Fast (graph walk)              |


In [ ]:
| Dataset Size              | Recommended Index                 |
| ------------------------- | --------------------------------- |
| UPTO 1L                     | `IndexFlatL2` or `IndexFlatIP`    |
| UPTO 1M                  | `IndexIVFFlat` or `IndexHNSWFlat` |
| > 1M                      | `IndexIVFPQ` or `IndexHNSWFlat`   |


## Section 5: Working with Documents and Metadata

Real documents usually have extra information attached, like source, author, date, etc. LangChain's `Document` object stores both `page_content` and `metadata`.

We will:
1. Create documents with metadata
2. Add them to a FAISS vector store
3. Search with and without metadata filters


In [ ]:
# from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [ ]:
documents

**Create a new FAISS index using Inner Product (IP).**

`IndexFlatIP` uses inner product instead of L2 distance. With normalized embeddings, inner product is equivalent to cosine similarity.


In [ ]:
index = faiss.IndexFlatIP(384)
vector_store = FAISS(
    embedding_function=embeddings,
    index= index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

**Add the documents to the vector store.**

`add_documents()` preserves metadata. Later, we can filter searches by metadata values.


In [ ]:
vector_store.add_documents(documents = documents)

**Add a few plain text strings without metadata.**

These will not have a `source` field, only content.


In [ ]:
vector_store.add_texts(["AI is future", "AI is powerful", "Dogs are cute", "Pizza is good", "I like mexican food"])

In [ ]:
vector_store.index_to_docstore_id

**Search without any filter.**

Returns the most similar documents regardless of source.


In [ ]:
vector_store.similarity_search("Langchain provides abstractions to make working with LLMs easy",
                               k=2) #Hyperparameter

**Search with metadata filter.**

Only return documents where `source == "news"`. This is useful when you want answers from a specific subset of your data.


In [ ]:
vector_store.similarity_search("Which animal is cute?",
                               k=2) #Hyperparameter

In [ ]:
vector_store.similarity_search(
    "Langchain provides abstractions to make working with LLMs easy",
    k=4,
    filter = {"source" : {"$eq": "news"}}
)

**Get similarity scores with results.**

`similarity_search_with_score()` returns tuples of `(document, score)`. The score helps you decide how confident the match is.


In [ ]:
vector_store.similarity_search_with_score("Langchain provides abstractions to make working with LLMs easy",
    k=4)

**Create a retriever with a score threshold.**

A retriever is a reusable component. This one only returns documents whose similarity score is at least `0.001`.


In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs = {
        "score_threshold" : 0.001,
        })

**Save the FAISS index to disk.**

`save_local()` writes the index and docstore to a folder. This lets you reuse the vector store without rebuilding it.


In [ ]:
vector_store.save_local("BIA FAISS Langchain")

**Load the saved FAISS index.**

`load_local()` reads the folder back. `allow_dangerous_deserialization=True` is needed because pickle files are used internally. Only load indexes you created yourself.


In [ ]:
new_vector_store = FAISS.load_local("BIA FAISS Langchain", embeddings,allow_dangerous_deserialization=True)

In [ ]:
new_vector_store.similarity_search("Langchain")

## Section 6: Retrievers, Scores & Thresholds

A **retriever** is a high-level interface to a vector store. You configure it once, then call `.invoke(query)` whenever you need results.

This section also shows how to filter results by score thresholds manually.


**Create a cosine-based vector store.**

`distance_strategy="COSINE"` makes the vector store use cosine similarity explicitly.


In [ ]:
vector_store_2 = FAISS.from_documents(
    documents, 
    embeddings, 
    distance_strategy="COSINE" # explicit cosine strategy
)



**Show help for `as_retriever()`.**

This Jupyter-specific command shows the method's documentation.


In [ ]:
vector_store.as_retriever?

**Create a simple similarity retriever.**

This retriever returns the top 2 most similar documents for any query.


In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

**Retrieve many documents with scores.**

Getting 15 results lets us apply our own filtering logic afterward.


In [ ]:
retrieved_documents = vector_store.similarity_search_with_score("Langchain provides abstractions to make working with LLMs easy",
    k=15)

**Inspect the first result.**

Each result is a tuple. Index `[0]` is the document, index `[1]` is the score.


In [ ]:
r_doc = retrieved_documents[0]
len(r_doc)

In [ ]:
r_doc

In [ ]:
r_doc[1]

In [ ]:
r_doc[0]

**Filter documents by a score threshold.**

Only keep documents with score >= 0.5. This is a simple way to avoid low-confidence matches.


In [ ]:
# Above 60%

doc_lis = []
thresh = 0.5
for r_doc in retrieved_documents:
    score = r_doc[1]
    if score >= thresh:
        doc_lis.append(r_doc[0])
    

In [ ]:
doc_lis

**Use relevance scores.**

`similarity_search_with_relevance_scores()` returns scores normalized to a 0-1 scale, where 1 means identical.


In [ ]:
vector_store.similarity_search_with_relevance_scores("Langchain provides abstractions to make working with LLMs easy",
    k=4)

**Invoke the retriever.**

Retrievers are commonly used in LangChain chains and RAG pipelines.


In [ ]:
retriever.invoke("Langchain provides abstractions to make working with LLMs easy")

In [ ]:
# Invoke the retriever
docs = retriever.invoke("LangChain projects")

for doc in docs:
    print(f"Content: {doc.page_content}")
    # Check if 'score' or 'relevance' exists in the metadata
    print(f"Metadata: {doc.metadata}")

## Section 7: ChromaDB

Chroma is another popular vector database. Unlike FAISS (in-memory by default), Chroma can persist data to disk using SQLite.

We will create a persistent Chroma collection using L2 distance and search it.


In [ ]:
# Chroma

**Install Chroma packages.**

Uncomment and run this cell if you haven't installed `langchain-chroma` and `chromadb` yet.


In [ ]:
!pip install langchain-chroma chromadb

**Import Chroma from LangChain.**


In [ ]:
from langchain_chroma import Chroma

**Create a persistent Chroma collection.**

- `collection_name`: name of the collection
- `persist_directory`: where Chroma stores its SQLite database
- `collection_metadata`: sets the distance function to L2

After running this, a `chroma_l2_db/` folder will appear.


In [ ]:
vector_store = Chroma.from_documents(
    documents = documents,
    embedding=embeddings,
    collection_name = "l2_collection",
    persist_directory="./chroma_l2_db",
    collection_metadata = {"hnsw:space" : "l2"}
)

**Create a retriever from Chroma.**


In [ ]:
retriever = vector_store.as_retriever(search_kwargs = {"k":2})

**Search the Chroma collection.**


In [ ]:
retriever.invoke("Langchain provides abstractions to make working with LLMs easy")